# AuraGateway vLLM CUDA 12.9 offline runtime compatibility verifier v7


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.parse
import zipfile
from datetime import UTC, datetime
from pathlib import Path, PurePosixPath
from typing import Any

NOTEBOOK_NAME = "auragateway-cu129-offline-verifier-v7"
INPUT_DIRECTORY_NAME = "auragateway_vllm_cu129_wheelhouse_v1"
OUTPUT_DIRECTORY_NAME = "auragateway_vllm_cu129_offline_compatibility_evidence_v7"
OUTPUT_ZIP = Path(f"/kaggle/working/{OUTPUT_DIRECTORY_NAME}.zip")
EVIDENCE_ROOT = Path(f"/kaggle/working/{OUTPUT_DIRECTORY_NAME}")
VENV_ROOT = Path("/kaggle/working/auragateway_vllm_runtime_cu129_v7")
MAX_EXCERPT = 16000

EXPECTED_PACKAGE_COUNT = 176
EXPECTED_MANIFEST_ENTRY_COUNT = 182
EXPECTED_NON_WHEEL_ENTRY_COUNT = 6
EXPECTED_TOTAL_WHEEL_BYTES = 5727339111
EXPECTED_NVJITLINK_SHA256 = (
    "02d3acb5fe598dd20f0fca3cc03734ad164037a22747a01900561a42d0b8448f"
)
EXPECTED_CUSPARSE_SHA256 = (
    "6d85ab1acdabfe3b5a54aa76bc948c719bcd03e07eede3eb8122b083c1a6ecf7"
)
REQUIRED_NVJITLINK_SYMBOL = "__nvJitLinkGetErrorLogSize_12_9"
DEPENDENCY_VALIDATION_POLICY = "CONTROLLED_TARGET_METADATA_AND_PACKAGING"

EXPECTED_HASHES = {
    "requirements.in": "a120c72a5643bb65afbfe0bd3dd072f1ea89a19f57a534dd814c9bafdd41880f",
    "resolution_lock.json": (
        "1575538b0a412c9b030fc95ccada0f0527553b76f06ef6b2b72904e61c84870c"
    ),
    "materialization.lock.txt": (
        "d061bd9a7ff0a686bb462a2bd016a1f3e1aea833fbdbff353dddf96fdd623e1d"
    ),
    "requirements.lock.txt": (
        "47cb357a53ca74ca597b286768e1d0e9cb831f7431c08fad378fc42ea59b3a27"
    ),
    "install_runtime.py": (
        "68bba3ca131e9a6f36392330562985d2a644be57cf5437fd282b883741c86821"
    ),
    "runtime_manifest.json": (
        "b424d2b952d726b2f7451ebd8f48d604985f650dbe2f6d146969625618b7fc51"
    ),
    "sha256_manifest.json": (
        "789fb23ab7d9c4f28dd909e808a53a65d692c0d7b43bc44da9e974817d771b8d"
    ),
    "materialization_receipt.json": (
        "52aa42b940dd606ab5685686ab893eb085efed2a7466989f654e870f4b360589"
    ),
}

EXPECTED_RUNTIME = {
    "python": "3.12",
    "cuda_variant": "cu129",
    "vllm_binary_cuda": "12.9",
    "vllm_release": "0.19.1",
    "vllm": "0.19.1",
    "torch": "2.10.0+cu129",
    "torchaudio": "2.10.0+cu129",
    "torchvision": "0.25.0+cu129",
    "transformers": "5.5.3",
}

EXPECTED_TOP_LEVEL = frozenset(
    {
        "requirements.in",
        "resolution_lock.json",
        "materialization.lock.txt",
        "requirements.lock.txt",
        "install_runtime.py",
        "runtime_manifest.json",
        "sha256_manifest.json",
        "materialization_receipt.json",
        "wheels",
    }
)

EXPECTED_MANIFEST_CONTROL_PATHS = frozenset(
    {
        "requirements.in",
        "resolution_lock.json",
        "materialization.lock.txt",
        "requirements.lock.txt",
        "install_runtime.py",
        "runtime_manifest.json",
    }
)

REQUIRED_ROLES = (
    "base_python_runtime",
    "base_venv_import",
    "base_pip_import",
    "base_distribution_snapshot_before",
    "gpu_topology",
    "venv_create_without_pip",
    "target_runtime_identity_before_install",
    "offline_hash_locked_install_via_base_pip_target_directory",
    "target_distribution_inventory",
    "target_dependency_check_via_controlled_python",
    "canonical_process_environment",
    "target_cuda_library_inventory",
    "inherited_nvjitlink_resolution",
    "canonical_nvjitlink_resolution",
    "canonical_nvjitlink_symbol",
    "canonical_cusparse_direct_load",
    "target_process_environment",
    "python_runtime",
    "torch_family_runtime",
    "transformers_runtime",
    "vllm_distribution",
    "vllm_module",
    "vllm_native_extension",
    "base_distribution_snapshot_after",
)

DISTRIBUTION_SNAPSHOT_SCRIPT = """
import hashlib
import importlib.metadata
import json
import re

items = sorted(
    (
        re.sub(r"[-_.]+", "-", str(dist.metadata.get("Name", ""))).lower(),
        dist.version,
    )
    for dist in importlib.metadata.distributions()
    if dist.metadata.get("Name")
)
encoded = json.dumps(
    items,
    ensure_ascii=True,
    separators=(",", ":"),
).encode("utf-8")
print(
    json.dumps(
        {
            "count": len(items),
            "sha256": hashlib.sha256(encoded).hexdigest(),
        }
    )
)
"""

TARGET_IDENTITY_SCRIPT = """
import importlib.util
import json
import platform
import site
import sys
import sysconfig
from pathlib import Path

expected = Path(sys.argv[1]).resolve()
prefix = Path(sys.prefix).resolve()
base_prefix = Path(sys.base_prefix).resolve()
paths = sysconfig.get_paths()
config = (prefix / "pyvenv.cfg").read_text(encoding="utf-8").lower()
target_site = Path(paths["purelib"]).resolve()


def module_origin(name):
    module = sys.modules.get(name)
    if module is None:
        return None
    return getattr(module, "__file__", None)


def external_package_paths():
    result = []
    for value in sys.path:
        if not value:
            continue
        path = Path(value).resolve()
        is_target = path == target_site or target_site in path.parents
        is_package_path = any(
            part in {"site-packages", "dist-packages"}
            for part in path.parts
        )
        if is_package_path and not is_target:
            result.append(value)
    return result


print(
    json.dumps(
        {
            "python": platform.python_version(),
            "prefix_matches_expected": prefix == expected,
            "base_prefix_differs": base_prefix != prefix,
            "user_site_enabled": site.ENABLE_USER_SITE,
            "purelib_within_prefix": target_site.is_relative_to(expected),
            "platlib_within_prefix": Path(paths["platlib"]).resolve().is_relative_to(
                expected
            ),
            "system_site_packages_enabled": (
                "include-system-site-packages = true" in config
            ),
            "pip_present": importlib.util.find_spec("pip") is not None,
            "no_site_flag": sys.flags.no_site,
            "sitecustomize_origin": module_origin("sitecustomize"),
            "usercustomize_origin": module_origin("usercustomize"),
            "external_package_paths": external_package_paths(),
        }
    )
)
"""

TARGET_INVENTORY_SCRIPT = """
import importlib.metadata
import json
import re

items = sorted(
    (
        re.sub(r"[-_.]+", "-", str(dist.metadata.get("Name", ""))).lower(),
        dist.version,
    )
    for dist in importlib.metadata.distributions()
    if dist.metadata.get("Name")
)
print(json.dumps(items, ensure_ascii=True, separators=(",", ":")))
"""

TARGET_DEPENDENCY_CHECK_SCRIPT = """
import importlib.metadata
import json
import sys
from pathlib import Path

from packaging.markers import default_environment
from packaging.requirements import Requirement
from packaging.utils import canonicalize_name

target_site = Path(sys.argv[1]).resolve()
distributions = tuple(
    importlib.metadata.distributions(path=[str(target_site)])
)
versions = {
    canonicalize_name(str(dist.metadata.get("Name", ""))): dist.version
    for dist in distributions
    if dist.metadata.get("Name")
}
environment = default_environment()
environment["extra"] = ""

missing = []
incompatible = []
invalid = []
evaluated_requirement_count = 0

for distribution in distributions:
    owner = canonicalize_name(
        str(distribution.metadata.get("Name", ""))
    )
    owner_version = distribution.version
    for raw_requirement in distribution.requires or ():
        try:
            requirement = Requirement(raw_requirement)
        except Exception as exc:
            invalid.append(
                {
                    "owner": owner,
                    "owner_version": owner_version,
                    "requirement": raw_requirement,
                    "error": type(exc).__name__,
                }
            )
            continue

        if (
            requirement.marker is not None
            and not requirement.marker.evaluate(environment)
        ):
            continue

        evaluated_requirement_count += 1
        dependency = canonicalize_name(requirement.name)
        installed_version = versions.get(dependency)
        if installed_version is None:
            missing.append(
                {
                    "owner": owner,
                    "requirement": str(requirement),
                    "dependency": dependency,
                }
            )
            continue

        if (
            requirement.specifier
            and not requirement.specifier.contains(
                installed_version,
                prereleases=True,
            )
        ):
            incompatible.append(
                {
                    "owner": owner,
                    "requirement": str(requirement),
                    "dependency": dependency,
                    "installed_version": installed_version,
                }
            )

passed = not missing and not incompatible and not invalid
print(
    json.dumps(
        {
            "status": "PASSED" if passed else "FAILED",
            "distribution_count": len(versions),
            "evaluated_requirement_count": evaluated_requirement_count,
            "missing": missing,
            "incompatible": incompatible,
            "invalid": invalid,
            "target_site": str(target_site),
        },
        ensure_ascii=True,
        sort_keys=True,
    )
)
raise SystemExit(0 if passed else 1)
"""

TARGET_PROCESS_SCRIPT = """
import json
import os
import site
import sys
from pathlib import Path

target_site = (
    Path(sys.prefix)
    / "lib"
    / f"python{sys.version_info.major}.{sys.version_info.minor}"
    / "site-packages"
).resolve()


def module_origin(name):
    module = sys.modules.get(name)
    if module is None:
        return None
    return getattr(module, "__file__", None)


external_package_paths = []
for value in sys.path:
    if not value:
        continue
    path = Path(value).resolve()
    is_target = path == target_site or target_site in path.parents
    is_package_path = any(
        part in {"site-packages", "dist-packages"}
        for part in path.parts
    )
    if is_package_path and not is_target:
        external_package_paths.append(value)


print(
    json.dumps(
        {
            "pythonpath_present": "PYTHONPATH" in os.environ,
            "pythonhome_present": "PYTHONHOME" in os.environ,
            "python_no_user_site": os.environ.get("PYTHONNOUSERSITE"),
            "user_site_enabled": site.ENABLE_USER_SITE,
            "sys_path": sys.path,
            "sitecustomize_origin": module_origin("sitecustomize"),
            "usercustomize_origin": module_origin("usercustomize"),
            "external_package_paths": external_package_paths,
            "no_site_flag": sys.flags.no_site,
        }
    )
)
"""

CONTROLLED_SITE_BOOTSTRAP_SCRIPT = """
import site
import sys
import types
from pathlib import Path

expected = Path(sys.argv[1]).resolve()
payload = sys.argv[2]
payload_args = sys.argv[3:]


def sentinel(name):
    module = types.ModuleType(name)
    module.__file__ = f"<auragateway-suppressed-{name}>"
    return module


sys.modules["sitecustomize"] = sentinel("sitecustomize")
sys.modules["usercustomize"] = sentinel("usercustomize")
site.main()

target_site = (
    expected
    / "lib"
    / f"python{sys.version_info.major}.{sys.version_info.minor}"
    / "site-packages"
).resolve()

cleaned = []
for value in sys.path:
    if not value:
        cleaned.append(value)
        continue
    path = Path(value).resolve()
    is_target = path == target_site or target_site in path.parents
    is_package_path = any(
        part in {"site-packages", "dist-packages"}
        for part in path.parts
    )
    if is_package_path and not is_target:
        continue
    cleaned.append(value)

if str(target_site) not in cleaned:
    cleaned.insert(0, str(target_site))

sys.path[:] = cleaned
sys.argv = ["<auragateway-target-probe>", *payload_args]
exec(
    compile(payload, "<auragateway-target-probe>", "exec"),
    {"__name__": "__main__"},
)
"""

_SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$")
_MATERIALIZATION_LINE = re.compile(
    r"^(?P<name>[^ ]+) @ (?P<url>\S+) --hash=sha256:(?P<sha>[0-9a-f]{64})$"
)
_REQUIREMENTS_LINE = re.compile(
    r"^(?P<name>[^= ]+)==(?P<version>\S+) "
    r"--hash=sha256:(?P<sha>[0-9a-f]{64})$"
)
_LDD_PATTERN = re.compile(
    r"libnvJitLink\.so\.12\s*=>\s*(?P<path>\S+)",
    re.IGNORECASE,
)


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def streaming_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def normalize_name(value: str) -> str:
    return re.sub(r"[-_.]+", "-", value).lower()


def sanitize(text: str | bytes | None) -> str:
    if text is None:
        return ""
    decoded = (
        text.decode("utf-8", errors="replace")
        if isinstance(text, bytes)
        else text
    )
    bounded = decoded[-MAX_EXCERPT:]
    replacements = {
        "/kaggle/input": "<input>",
        "/kaggle/working": "<working>",
        os.environ.get("HOME", ""): "<home>",
    }
    for source, replacement in replacements.items():
        if source:
            bounded = bounded.replace(source, replacement)
    return bounded


def unique_paths(values: list[str]) -> list[str]:
    observed: set[str] = set()
    result: list[str] = []
    for value in values:
        if not value or value in observed:
            continue
        observed.add(value)
        result.append(value)
    return result


def base_offline_environment() -> dict[str, str]:
    environment = {**os.environ}
    environment.pop("PYTHONPATH", None)
    environment.pop("PYTHONHOME", None)
    environment["PYTHONNOUSERSITE"] = "1"
    environment["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
    environment["PIP_NO_INDEX"] = "1"
    environment["HF_HUB_OFFLINE"] = "1"
    environment["TRANSFORMERS_OFFLINE"] = "1"
    return environment


def controlled_target_argv(
    python: Path,
    payload: str,
    *payload_args: str,
) -> list[str]:
    return [
        str(python),
        "-S",
        "-c",
        CONTROLLED_SITE_BOOTSTRAP_SCRIPT,
        str(VENV_ROOT),
        payload,
        *payload_args,
    ]


def canonical_target_environment(
    target_library_directories: list[Path],
) -> tuple[dict[str, str], dict[str, object]]:
    environment = base_offline_environment()
    inherited = [
        item
        for item in os.environ.get("LD_LIBRARY_PATH", "").split(os.pathsep)
        if item
    ]
    target = [str(path.resolve()) for path in target_library_directories]
    ordered = unique_paths(target + inherited)
    environment["LD_LIBRARY_PATH"] = os.pathsep.join(ordered)
    environment["VIRTUAL_ENV"] = str(VENV_ROOT)
    environment["PATH"] = os.pathsep.join(
        unique_paths([str(VENV_ROOT / "bin"), *environment["PATH"].split(os.pathsep)])
    )
    report = {
        "schema_version": "1.0.0",
        "status": "PASSED",
        "command_role": "canonical_process_environment",
        "policy": "TARGET_NVIDIA_LIBRARIES_PREPENDED",
        "python_startup_policy": "NO_SITE_WITH_CONTROLLED_SITE_BOOTSTRAP",
        "pythonpath_removed": "PYTHONPATH" not in environment,
        "pythonhome_removed": "PYTHONHOME" not in environment,
        "python_no_user_site": environment["PYTHONNOUSERSITE"],
        "target_library_directories": [sanitize(item) for item in target],
        "inherited_library_directories": [
            sanitize(item) for item in unique_paths(inherited)
        ],
        "effective_library_directories": [
            sanitize(item) for item in ordered
        ],
        "returncode": 0,
        "timed_out": False,
        "stdout_excerpt": "",
        "stderr_excerpt": "",
    }
    return environment, report


def run_probe(
    role: str,
    argv: list[str],
    *,
    timeout: float,
    environment: dict[str, str] | None = None,
) -> dict[str, object]:
    started = time.monotonic()
    started_at = datetime.now(UTC).isoformat(timespec="seconds")
    env = base_offline_environment() if environment is None else environment
    try:
        result = subprocess.run(
            argv,
            check=False,
            capture_output=True,
            text=True,
            timeout=timeout,
            env=env,
        )
        return {
            "schema_version": "3.0.0",
            "command_role": role,
            "argv": [sanitize(item) for item in argv],
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": result.returncode,
            "timed_out": False,
            "status": "PASSED" if result.returncode == 0 else "FAILED",
            "stdout_excerpt": sanitize(result.stdout),
            "stderr_excerpt": sanitize(result.stderr),
        }
    except subprocess.TimeoutExpired as exc:
        return {
            "schema_version": "3.0.0",
            "command_role": role,
            "argv": [sanitize(item) for item in argv],
            "started_at": started_at,
            "duration_ms": int((time.monotonic() - started) * 1000),
            "returncode": None,
            "timed_out": True,
            "status": "FAILED",
            "stdout_excerpt": sanitize(exc.stdout),
            "stderr_excerpt": sanitize(exc.stderr),
        }


def blocked_record(
    role: str,
    *,
    blocked_by: str,
    reason: str,
) -> dict[str, object]:
    return {
        "schema_version": "3.0.0",
        "command_role": role,
        "status": "BLOCKED_BY_UPSTREAM_FAILURE",
        "blocked_by": blocked_by,
        "reason": reason,
        "returncode": None,
        "timed_out": False,
        "stdout_excerpt": "",
        "stderr_excerpt": "",
    }


def not_executed_record(
    role: str,
    *,
    reason: str,
) -> dict[str, object]:
    return {
        "schema_version": "3.0.0",
        "command_role": role,
        "status": "NOT_EXECUTED",
        "reason": reason,
        "returncode": None,
        "timed_out": False,
        "stdout_excerpt": "",
        "stderr_excerpt": "",
    }


def summarize_role_states(
    records: list[dict[str, object]],
) -> dict[str, list[str]]:
    observed = {str(record["command_role"]): record for record in records}
    for role in REQUIRED_ROLES:
        if role not in observed:
            record = not_executed_record(
                role,
                reason="role record was not emitted",
            )
            records.append(record)
            observed[role] = record
    return {
        "failed_required_roles": sorted(
            role
            for role in REQUIRED_ROLES
            if observed[role]["status"] == "FAILED"
        ),
        "blocked_required_roles": sorted(
            role
            for role in REQUIRED_ROLES
            if observed[role]["status"] == "BLOCKED_BY_UPSTREAM_FAILURE"
        ),
        "not_executed_required_roles": sorted(
            role
            for role in REQUIRED_ROLES
            if observed[role]["status"] == "NOT_EXECUTED"
        ),
    }


def require_json_object(path: Path) -> dict[str, Any]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError(f"expected one JSON object: {path.name}")
    return payload


def validate_control_hashes(input_root: Path) -> None:
    for relative, expected in EXPECTED_HASHES.items():
        path = input_root / relative
        if not path.is_file() or path.is_symlink():
            raise RuntimeError(
                f"control file is missing or unsafe: {relative}"
            )
        if streaming_sha256(path) != expected:
            raise RuntimeError(
                f"control file identity drifted: {relative}"
            )


def parse_manifest_entries(
    payload: dict[str, Any],
) -> list[dict[str, Any]]:
    if payload.get("schema_version") != "1.0.0":
        raise RuntimeError(
            "wheelhouse SHA-256 manifest schema drifted"
        )
    entries = payload.get("entries")
    if (
        not isinstance(entries, list)
        or len(entries) != EXPECTED_MANIFEST_ENTRY_COUNT
    ):
        raise RuntimeError(
            "wheelhouse SHA-256 manifest entry count drifted"
        )
    parsed: list[dict[str, Any]] = []
    for entry in entries:
        if not isinstance(entry, dict):
            raise RuntimeError(
                "wheelhouse SHA-256 entry is invalid"
            )
        path = entry.get("path")
        digest = entry.get("sha256")
        size_bytes = entry.get("size_bytes")
        if (
            not isinstance(path, str)
            or not isinstance(digest, str)
            or _SHA256_PATTERN.fullmatch(digest) is None
            or not isinstance(size_bytes, int)
            or size_bytes < 0
        ):
            raise RuntimeError(
                "wheelhouse SHA-256 entry fields are invalid"
            )
        relative = PurePosixPath(path)
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError(
                "wheelhouse SHA-256 path is unsafe"
            )
        parsed.append(
            {
                "path": path,
                "sha256": digest,
                "size_bytes": size_bytes,
            }
        )
    paths = [str(entry["path"]) for entry in parsed]
    hashes = [str(entry["sha256"]) for entry in parsed]
    if len(paths) != len(set(paths)):
        raise RuntimeError(
            "wheelhouse SHA-256 paths are not unique"
        )
    if len(hashes) != len(set(hashes)):
        raise RuntimeError(
            "wheelhouse SHA-256 identities are not unique"
        )
    return parsed


def validate_manifest_files(
    input_root: Path,
    entries: list[dict[str, Any]],
) -> None:
    for entry in entries:
        relative = PurePosixPath(str(entry["path"]))
        path = input_root.joinpath(*relative.parts)
        if not path.is_file() or path.is_symlink():
            raise RuntimeError(
                f"wheelhouse payload member is missing or unsafe: "
                f"{relative}"
            )
        if path.stat().st_size != entry["size_bytes"]:
            raise RuntimeError(
                f"wheelhouse payload size drifted: {relative}"
            )
        if streaming_sha256(path) != entry["sha256"]:
            raise RuntimeError(
                f"wheelhouse payload identity drifted: {relative}"
            )


def parse_materialization_lock(
    path: Path,
) -> dict[str, dict[str, str]]:
    lines = [
        line.strip()
        for line in path.read_text(
            encoding="utf-8",
        ).splitlines()
    ]
    lines = [line for line in lines if line]
    if len(lines) != EXPECTED_PACKAGE_COUNT:
        raise RuntimeError(
            "materialization lock package count drifted"
        )
    parsed: dict[str, dict[str, str]] = {}
    for line in lines:
        match = _MATERIALIZATION_LINE.fullmatch(line)
        if match is None:
            raise RuntimeError(
                "materialization lock line is invalid"
            )
        record = match.groupdict()
        key = normalize_name(record["name"])
        if key in parsed:
            raise RuntimeError(
                "materialization lock package names are not unique"
            )
        parsed[key] = record
    return parsed


def parse_requirements_lock(
    path: Path,
) -> dict[str, dict[str, str]]:
    lines = [
        line.strip()
        for line in path.read_text(
            encoding="utf-8",
        ).splitlines()
    ]
    lines = [line for line in lines if line]
    if len(lines) != EXPECTED_PACKAGE_COUNT:
        raise RuntimeError(
            "requirements lock package count drifted"
        )
    parsed: dict[str, dict[str, str]] = {}
    for line in lines:
        match = _REQUIREMENTS_LINE.fullmatch(line)
        if match is None:
            raise RuntimeError(
                "requirements lock line is invalid"
            )
        record = match.groupdict()
        key = normalize_name(record["name"])
        if key in parsed:
            raise RuntimeError(
                "requirements lock package names are not unique"
            )
        parsed[key] = record
    return parsed


def validate_resolution_closure(
    input_root: Path,
    entries: list[dict[str, Any]],
) -> dict[str, object]:
    resolution = require_json_object(
        input_root / "resolution_lock.json"
    )
    if (
        resolution.get("schema_version") != "1.0.0"
        or resolution.get("package_count")
        != EXPECTED_PACKAGE_COUNT
        or resolution.get("host_count") != 5
        or resolution.get("wildcard_domains_permitted")
        is not False
        or resolution.get("review_decision")
        != "APPROVED_AS_EXACT_LOCKED_CLOSURE"
    ):
        raise RuntimeError(
            "resolution lock contract drifted"
        )
    records = resolution.get("records")
    if (
        not isinstance(records, list)
        or len(records) != EXPECTED_PACKAGE_COUNT
    ):
        raise RuntimeError(
            "resolution lock records drifted"
        )

    entry_by_path = {
        str(entry["path"]): entry
        for entry in entries
    }
    wheel_entries = {
        path: entry
        for path, entry in entry_by_path.items()
        if path.startswith("wheels/")
    }
    control_paths = set(entry_by_path) - set(wheel_entries)
    if len(wheel_entries) != EXPECTED_PACKAGE_COUNT:
        raise RuntimeError(
            "wheel manifest package count drifted"
        )
    if control_paths != EXPECTED_MANIFEST_CONTROL_PATHS:
        raise RuntimeError(
            "wheelhouse control manifest topology drifted"
        )

    materialization_lock = parse_materialization_lock(
        input_root / "materialization.lock.txt"
    )
    requirements_lock = parse_requirements_lock(
        input_root / "requirements.lock.txt"
    )

    observed_names: set[str] = set()
    observed_wheel_paths: set[str] = set()
    for record in records:
        if not isinstance(record, dict):
            raise RuntimeError(
                "resolution lock record is invalid"
            )
        name = record.get("normalized_name")
        version = record.get("version")
        artifact_filename = record.get(
            "artifact_filename"
        )
        artifact_sha256 = record.get("sha256")
        sanitized_url = record.get("sanitized_url")
        if not all(
            isinstance(value, str)
            for value in (
                name,
                version,
                artifact_filename,
                artifact_sha256,
                sanitized_url,
            )
        ):
            raise RuntimeError(
                "resolution lock record fields are invalid"
            )
        key = normalize_name(name)
        if key in observed_names:
            raise RuntimeError(
                "resolution lock package names are not unique"
            )
        observed_names.add(key)

        decoded_filename = urllib.parse.unquote(
            artifact_filename
        )
        wheel_path = f"wheels/{decoded_filename}"
        observed_wheel_paths.add(wheel_path)
        manifest_entry = wheel_entries.get(wheel_path)
        if (
            manifest_entry is None
            or manifest_entry["sha256"]
            != artifact_sha256
        ):
            raise RuntimeError(
                f"resolution-to-wheel identity drifted: {key}"
            )

        materialized = materialization_lock.get(key)
        if (
            materialized is None
            or materialized["url"] != sanitized_url
            or materialized["sha"]
            != artifact_sha256
        ):
            raise RuntimeError(
                f"materialization lock drifted: {key}"
            )

        requirement = requirements_lock.get(key)
        if (
            requirement is None
            or requirement["version"] != version
            or requirement["sha"]
            != artifact_sha256
        ):
            raise RuntimeError(
                f"requirements lock drifted: {key}"
            )

    if observed_wheel_paths != set(wheel_entries):
        raise RuntimeError(
            "wheel manifest contains unexpected package artifacts"
        )

    total_wheel_bytes = sum(
        int(entry["size_bytes"])
        for entry in wheel_entries.values()
    )
    if total_wheel_bytes != EXPECTED_TOTAL_WHEEL_BYTES:
        raise RuntimeError(
            "wheelhouse byte count drifted"
        )

    return {
        "manifest_entry_count": len(entries),
        "wheel_entry_count": len(wheel_entries),
        "non_wheel_entry_count": len(control_paths),
        "total_wheel_bytes": total_wheel_bytes,
        "package_count": len(observed_names),
    }


def validate_receipt(input_root: Path) -> None:
    receipt = require_json_object(
        input_root / "materialization_receipt.json"
    )
    expected = {
        "schema_version": "1.2.0",
        "notebook_name": (
            "auragateway-cu129-wheelhouse-materializer-v1"
        ),
        "output_directory": INPUT_DIRECTORY_NAME,
        "materialization_status": "PASSED",
        "package_count": EXPECTED_PACKAGE_COUNT,
        "total_wheel_bytes": EXPECTED_TOTAL_WHEEL_BYTES,
        "resolution_lock_sha256": EXPECTED_HASHES[
            "resolution_lock.json"
        ],
        "materialization_lock_sha256": EXPECTED_HASHES[
            "materialization.lock.txt"
        ],
        "requirements_lock_sha256": EXPECTED_HASHES[
            "requirements.lock.txt"
        ],
        "runtime_manifest_sha256": EXPECTED_HASHES[
            "runtime_manifest.json"
        ],
        "sha256_manifest_sha256": EXPECTED_HASHES[
            "sha256_manifest.json"
        ],
        "pip_download_subcommand_performed": True,
        "pip_resolution_artifact_transfer_observed": True,
        "pip_resolution_transfer_event_count": 358,
        "package_installation_performed": False,
        "model_requests_performed": 0,
        "qualification_claimed": False,
        "credentials_used": False,
        "customer_data_used": False,
    }
    drift = sorted(
        key
        for key, value in expected.items()
        if receipt.get(key) != value
    )
    if drift:
        raise RuntimeError(
            "materialization receipt drifted: "
            + ", ".join(drift)
        )


def validate_runtime_manifest(input_root: Path) -> None:
    runtime = require_json_object(
        input_root / "runtime_manifest.json"
    )
    for key, value in EXPECTED_RUNTIME.items():
        if runtime.get(key) != value:
            raise RuntimeError(
                f"runtime manifest drifted at {key}"
            )
    if (
        runtime.get("schema_version") != "1.2.0"
        or runtime.get("package_count")
        != EXPECTED_PACKAGE_COUNT
        or runtime.get(
            "network_required_for_installation"
        )
        is not False
        or runtime.get("model_requests_performed") != 0
        or runtime.get("qualification_claimed") is not False
    ):
        raise RuntimeError(
            "runtime manifest safety contract drifted"
        )


def validate_input(
    input_root: Path,
) -> dict[str, object]:
    if (
        not input_root.is_dir()
        or input_root.is_symlink()
    ):
        raise RuntimeError(
            "governed wheelhouse root is missing or unsafe"
        )
    observed_topology = {
        path.name
        for path in input_root.iterdir()
    }
    if observed_topology != EXPECTED_TOP_LEVEL:
        raise RuntimeError(
            "wheelhouse top-level topology drifted"
        )

    validate_control_hashes(input_root)
    sha_manifest = require_json_object(
        input_root / "sha256_manifest.json"
    )
    entries = parse_manifest_entries(sha_manifest)
    validate_manifest_files(input_root, entries)
    closure = validate_resolution_closure(
        input_root,
        entries,
    )
    validate_receipt(input_root)
    validate_runtime_manifest(input_root)

    free_bytes = shutil.disk_usage(
        "/kaggle/working"
    ).free
    if free_bytes < 12 * 1024**3:
        raise RuntimeError(
            "insufficient writable disk for isolated runtime"
        )

    return {
        "schema_version": "3.0.0",
        "status": "PASSED",
        "input_directory_name": INPUT_DIRECTORY_NAME,
        "input_sha256_manifest_sha256": EXPECTED_HASHES[
            "sha256_manifest.json"
        ],
        "input_materialization_receipt_sha256": (
            EXPECTED_HASHES[
                "materialization_receipt.json"
            ]
        ),
        "input_runtime_manifest_sha256": EXPECTED_HASHES[
            "runtime_manifest.json"
        ],
        "resolution_lock_sha256": EXPECTED_HASHES[
            "resolution_lock.json"
        ],
        "package_count": closure["package_count"],
        "manifest_entry_count": closure[
            "manifest_entry_count"
        ],
        "wheel_entry_count": closure[
            "wheel_entry_count"
        ],
        "non_wheel_entry_count": closure[
            "non_wheel_entry_count"
        ],
        "total_wheel_bytes": closure[
            "total_wheel_bytes"
        ],
        "working_free_bytes_before_install": free_bytes,
        "network_access_requested": False,
        "credentials_used": False,
        "customer_data_used": False,
        "model_requests_performed": 0,
        "qualification_claimed": False,
    }


def expected_distribution_inventory(
    input_root: Path,
) -> list[list[str]]:
    resolution = require_json_object(
        input_root / "resolution_lock.json"
    )
    records = resolution.get("records")
    if not isinstance(records, list):
        raise RuntimeError(
            "resolution lock records are missing"
        )
    inventory: list[list[str]] = []
    for record in records:
        if not isinstance(record, dict):
            raise RuntimeError(
                "resolution lock record is invalid"
            )
        name = record.get("normalized_name")
        version = record.get("version")
        if (
            not isinstance(name, str)
            or not isinstance(version, str)
        ):
            raise RuntimeError(
                "resolution lock inventory fields are invalid"
            )
        inventory.append(
            [normalize_name(name), version]
        )
    return sorted(inventory)


def parse_json_stdout(
    record: dict[str, object],
) -> object | None:
    try:
        return json.loads(
            str(record["stdout_excerpt"]).strip()
        )
    except json.JSONDecodeError:
        return None


def pip_version_supported(value: str) -> bool:
    match = re.match(r"^(\d+)\.(\d+)", value)
    if match is None:
        return False
    return (
        int(match.group(1)),
        int(match.group(2)),
    ) >= (22, 3)


def target_site_packages() -> Path:
    return (
        VENV_ROOT
        / "lib"
        / f"python{sys.version_info.major}.{sys.version_info.minor}"
        / "site-packages"
    )


def target_nvidia_library_directories(
    site_packages: Path,
) -> list[Path]:
    root = site_packages / "nvidia"
    discovered = sorted(
        path.resolve()
        for path in root.glob("*/lib")
        if path.is_dir()
    )
    priorities = (
        root / "nvjitlink" / "lib",
        root / "cusparse" / "lib",
        root / "cuda_runtime" / "lib",
    )
    ordered = [
        path.resolve()
        for path in priorities
        if path.is_dir()
    ]
    ordered.extend(discovered)
    return [
        Path(value)
        for value in unique_paths(
            [str(path) for path in ordered]
        )
    ]


def library_inventory_record(
    site_packages: Path,
    directories: list[Path],
) -> dict[str, object]:
    nvjitlink = (
        site_packages
        / "nvidia"
        / "nvjitlink"
        / "lib"
        / "libnvJitLink.so.12"
    )
    cusparse = (
        site_packages
        / "nvidia"
        / "cusparse"
        / "lib"
        / "libcusparse.so.12"
    )
    if not nvjitlink.is_file() or not cusparse.is_file():
        raise RuntimeError(
            "target CUDA libraries are missing"
        )
    return {
        "schema_version": "1.0.0",
        "command_role": "target_cuda_library_inventory",
        "status": "PASSED",
        "target_library_directories": [
            sanitize(str(path))
            for path in directories
        ],
        "libnvJitLink": {
            "path": sanitize(str(nvjitlink)),
            "realpath": sanitize(
                str(nvjitlink.resolve())
            ),
            "sha256": streaming_sha256(nvjitlink),
            "size_bytes": nvjitlink.stat().st_size,
        },
        "libcusparse": {
            "path": sanitize(str(cusparse)),
            "realpath": sanitize(
                str(cusparse.resolve())
            ),
            "sha256": streaming_sha256(cusparse),
            "size_bytes": cusparse.stat().st_size,
        },
        "returncode": 0,
        "timed_out": False,
        "stdout_excerpt": "",
        "stderr_excerpt": "",
    }


def ldd_resolved_path(
    record: dict[str, object],
) -> str | None:
    text = (
        str(record.get("stdout_excerpt", ""))
        + "\n"
        + str(record.get("stderr_excerpt", ""))
    )
    match = _LDD_PATTERN.search(text)
    if match is None:
        return None
    return match.group("path")


def reject_semantic(
    observed: dict[str, dict[str, object]],
    role: str,
    message: str,
) -> None:
    record = observed.get(role)
    if (
        record is None
        or record["status"] != "PASSED"
    ):
        return
    record["status"] = "FAILED"
    record["semantic_error"] = message


def apply_semantic_checks(
    records: list[dict[str, object]],
    input_root: Path | None,
) -> None:
    observed = {
        str(record["command_role"]): record
        for record in records
    }

    base_python = observed.get(
        "base_python_runtime"
    )
    if (
        base_python is not None
        and base_python["status"] == "PASSED"
        and not str(
            base_python["stdout_excerpt"]
        ).strip().startswith("3.12.")
    ):
        reject_semantic(
            observed,
            "base_python_runtime",
            "base Python is not 3.12.x",
        )

    base_venv = observed.get("base_venv_import")
    if (
        base_venv is not None
        and base_venv["status"] == "PASSED"
        and str(
            base_venv["stdout_excerpt"]
        ).strip()
        != "ok"
    ):
        reject_semantic(
            observed,
            "base_venv_import",
            "venv import did not emit ok",
        )

    base_pip = observed.get("base_pip_import")
    if (
        base_pip is not None
        and base_pip["status"] == "PASSED"
    ):
        payload = parse_json_stdout(base_pip)
        if not isinstance(payload, dict):
            reject_semantic(
                observed,
                "base_pip_import",
                "base pip output is not JSON",
            )
        else:
            version = payload.get("version")
            if (
                not isinstance(version, str)
                or not pip_version_supported(version)
            ):
                reject_semantic(
                    observed,
                    "base_pip_import",
                    "base pip lacks --python support",
                )

    topology = observed.get("gpu_topology")
    if (
        topology is not None
        and topology["status"] == "PASSED"
    ):
        lines = [
            line.strip()
            for line in str(
                topology["stdout_excerpt"]
            ).splitlines()
            if line.strip()
        ]
        if len(lines) != 2:
            reject_semantic(
                observed,
                "gpu_topology",
                "expected exactly two GPU records",
            )
        else:
            for index, line in enumerate(lines):
                parts = [
                    part.strip()
                    for part in line.split(",")
                ]
                if (
                    len(parts) != 5
                    or parts[0] != str(index)
                    or parts[1]
                    not in {"Tesla T4", "NVIDIA T4"}
                    or parts[3] != "7.5"
                ):
                    reject_semantic(
                        observed,
                        "gpu_topology",
                        "GPU topology is not T4 x2",
                    )
                    break

    identity = observed.get(
        "target_runtime_identity_before_install"
    )
    if (
        identity is not None
        and identity["status"] == "PASSED"
    ):
        payload = parse_json_stdout(identity)
        expected_identity = {
            "python": "3.12.13",
            "prefix_matches_expected": True,
            "base_prefix_differs": True,
            "user_site_enabled": False,
            "purelib_within_prefix": True,
            "platlib_within_prefix": True,
            "system_site_packages_enabled": False,
            "pip_present": False,
            "no_site_flag": 1,
            "sitecustomize_origin": "<auragateway-suppressed-sitecustomize>",
            "usercustomize_origin": "<auragateway-suppressed-usercustomize>",
            "external_package_paths": [],
        }
        if payload != expected_identity:
            reject_semantic(
                observed,
                "target_runtime_identity_before_install",
                "target environment identity drifted",
            )

    inventory = observed.get(
        "target_distribution_inventory"
    )
    if (
        inventory is not None
        and inventory["status"] == "PASSED"
        and input_root is not None
        and parse_json_stdout(inventory)
        != expected_distribution_inventory(input_root)
    ):
        reject_semantic(
            observed,
            "target_distribution_inventory",
            "target closure drifted",
        )

    environment = observed.get(
        "canonical_process_environment"
    )
    if (
        environment is not None
        and environment["status"] == "PASSED"
        and (
            environment.get("policy")
            != "TARGET_NVIDIA_LIBRARIES_PREPENDED"
            or environment.get("pythonpath_removed")
            is not True
            or environment.get("pythonhome_removed")
            is not True
            or environment.get("python_no_user_site")
            != "1"
            or environment.get("python_startup_policy")
            != "NO_SITE_WITH_CONTROLLED_SITE_BOOTSTRAP"
        )
    ):
        reject_semantic(
            observed,
            "canonical_process_environment",
            "canonical process environment drifted",
        )

    library_inventory = observed.get(
        "target_cuda_library_inventory"
    )
    if (
        library_inventory is not None
        and library_inventory["status"] == "PASSED"
    ):
        nvjitlink = library_inventory.get(
            "libnvJitLink"
        )
        cusparse = library_inventory.get(
            "libcusparse"
        )
        if (
            not isinstance(nvjitlink, dict)
            or nvjitlink.get("sha256")
            != EXPECTED_NVJITLINK_SHA256
            or not isinstance(cusparse, dict)
            or cusparse.get("sha256")
            != EXPECTED_CUSPARSE_SHA256
        ):
            reject_semantic(
                observed,
                "target_cuda_library_inventory",
                "target CUDA library identity drifted",
            )

    canonical_resolution = observed.get(
        "canonical_nvjitlink_resolution"
    )
    if (
        canonical_resolution is not None
        and canonical_resolution["status"] == "PASSED"
    ):
        resolved = ldd_resolved_path(
            canonical_resolution
        )
        if (
            resolved is None
            or "<working>/auragateway_vllm_runtime_cu129_v6/"
            not in resolved
            or "/nvidia/nvjitlink/lib/"
            not in resolved
        ):
            reject_semantic(
                observed,
                "canonical_nvjitlink_resolution",
                "canonical loader did not select target nvJitLink",
            )

    symbol = observed.get(
        "canonical_nvjitlink_symbol"
    )
    if (
        symbol is not None
        and symbol["status"] == "PASSED"
        and REQUIRED_NVJITLINK_SYMBOL
        not in str(symbol["stdout_excerpt"])
    ):
        reject_semantic(
            observed,
            "canonical_nvjitlink_symbol",
            "required CUDA 12.9 symbol is absent",
        )

    direct_load = observed.get(
        "canonical_cusparse_direct_load"
    )
    if (
        direct_load is not None
        and direct_load["status"] == "PASSED"
        and str(
            direct_load["stdout_excerpt"]
        ).strip()
        != "loaded"
    ):
        reject_semantic(
            observed,
            "canonical_cusparse_direct_load",
            "canonical cuSPARSE direct load drifted",
        )

    target_process = observed.get(
        "target_process_environment"
    )
    if (
        target_process is not None
        and target_process["status"] == "PASSED"
    ):
        payload = parse_json_stdout(target_process)
        if (
            not isinstance(payload, dict)
            or payload.get("pythonpath_present")
            is not False
            or payload.get("pythonhome_present")
            is not False
            or payload.get("python_no_user_site")
            != "1"
            or payload.get("user_site_enabled")
            is not False
            or payload.get("sitecustomize_origin")
            != "<auragateway-suppressed-sitecustomize>"
            or payload.get("usercustomize_origin")
            != "<auragateway-suppressed-usercustomize>"
            or payload.get("external_package_paths") != []
            or payload.get("no_site_flag") != 1
        ):
            reject_semantic(
                observed,
                "target_process_environment",
                "target process inherited base Python paths",
            )

    python_runtime = observed.get("python_runtime")
    if (
        python_runtime is not None
        and python_runtime["status"] == "PASSED"
        and not str(
            python_runtime["stdout_excerpt"]
        ).strip().startswith("3.12.")
    ):
        reject_semantic(
            observed,
            "python_runtime",
            "target Python is not 3.12.x",
        )

    torch_runtime = observed.get(
        "torch_family_runtime"
    )
    if (
        torch_runtime is not None
        and torch_runtime["status"] == "PASSED"
    ):
        payload = parse_json_stdout(torch_runtime)
        expected_torch = {
            "torch": "2.10.0+cu129",
            "torchaudio": "2.10.0+cu129",
            "torchvision": "0.25.0+cu129",
            "cuda": "12.9",
            "available": True,
            "device_count": 2,
        }
        if payload != expected_torch:
            reject_semantic(
                observed,
                "torch_family_runtime",
                "Torch or CUDA runtime drifted",
            )

    dependency_check = observed.get(
        "target_dependency_check_via_controlled_python"
    )
    if (
        dependency_check is not None
        and dependency_check["status"] == "PASSED"
    ):
        payload = parse_json_stdout(dependency_check)
        if (
            not isinstance(payload, dict)
            or payload.get("status") != "PASSED"
            or payload.get("distribution_count")
            != EXPECTED_PACKAGE_COUNT
            or payload.get("missing") != []
            or payload.get("incompatible") != []
            or payload.get("invalid") != []
        ):
            reject_semantic(
                observed,
                "target_dependency_check_via_controlled_python",
                "target dependency closure drifted",
            )

    expected_stdout = {
        "transformers_runtime": "5.5.3",
        "vllm_distribution": "0.19.1",
        "vllm_module": "0.19.1",
        "vllm_native_extension": "ok",
    }
    for role, expected_value in expected_stdout.items():
        record = observed.get(role)
        if (
            record is not None
            and record["status"] == "PASSED"
            and str(
                record["stdout_excerpt"]
            ).strip()
            != expected_value
        ):
            reject_semantic(
                observed,
                role,
                f"expected {expected_value}",
            )

    for role in (
        "target_runtime_identity_before_install",
        "target_distribution_inventory",
        "target_dependency_check_via_controlled_python",
        "target_process_environment",
        "python_runtime",
        "torch_family_runtime",
        "transformers_runtime",
        "vllm_distribution",
        "vllm_module",
        "vllm_native_extension",
    ):
        record = observed.get(role)
        if (
            record is not None
            and record["status"] == "PASSED"
            and "sitecustomize" in str(
                record.get("stderr_excerpt", "")
            ).lower()
        ):
            reject_semantic(
                observed,
                role,
                "base sitecustomize leaked into target process",
            )

    before = observed.get(
        "base_distribution_snapshot_before"
    )
    after = observed.get(
        "base_distribution_snapshot_after"
    )
    if (
        before is not None
        and after is not None
        and before["status"] == "PASSED"
        and after["status"] == "PASSED"
        and parse_json_stdout(before)
        != parse_json_stdout(after)
    ):
        reject_semantic(
            observed,
            "base_distribution_snapshot_after",
            "base distribution metadata changed",
        )


def dependency_failure(
    records: list[dict[str, object]],
    dependencies: tuple[str, ...],
) -> str | None:
    observed = {
        str(record["command_role"]): record
        for record in records
    }
    for dependency in dependencies:
        record = observed.get(dependency)
        if (
            record is None
            or record["status"] != "PASSED"
        ):
            return dependency
    return None


def append_dependent_probe(
    records: list[dict[str, object]],
    role: str,
    dependencies: tuple[str, ...],
    argv: list[str],
    *,
    timeout: float,
    blocked_reason: str,
    environment: dict[str, str] | None = None,
) -> None:
    failed_dependency = dependency_failure(
        records,
        dependencies,
    )
    if failed_dependency is not None:
        records.append(
            blocked_record(
                role,
                blocked_by=failed_dependency,
                reason=blocked_reason,
            )
        )
        return
    records.append(
        run_probe(
            role,
            argv,
            timeout=timeout,
            environment=environment,
        )
    )


def write_evidence_record(
    name: str,
    payload: object,
) -> None:
    (EVIDENCE_ROOT / name).write_text(
        canonical_json(payload) + "\n",
        encoding="utf-8",
    )


def main() -> int:
    if (
        os.environ.get(
            "AURAGATEWAY_CUSTOMER_DATA_PRESENT"
        )
        == "1"
    ):
        raise RuntimeError(
            "customer data is prohibited"
        )

    if EVIDENCE_ROOT.exists():
        shutil.rmtree(EVIDENCE_ROOT)
    EVIDENCE_ROOT.mkdir(parents=True)
    if VENV_ROOT.exists():
        shutil.rmtree(VENV_ROOT)
    if OUTPUT_ZIP.exists():
        OUTPUT_ZIP.unlink()

    records: list[dict[str, object]] = []
    input_validation: dict[str, object]
    input_root: Path | None = None
    matches = tuple(
        Path("/kaggle/input").rglob(
            INPUT_DIRECTORY_NAME
        )
    )

    try:
        if len(matches) != 1:
            raise RuntimeError(
                "expected exactly one governed wheelhouse directory"
            )
        input_root = matches[0]
        input_validation = validate_input(
            input_root
        )
    except Exception as exc:
        input_validation = {
            "schema_version": "3.0.0",
            "status": "FAILED",
            "error_type": type(exc).__name__,
            "error_message": sanitize(str(exc)),
            "network_access_requested": False,
            "credentials_used": False,
            "customer_data_used": False,
            "model_requests_performed": 0,
            "qualification_claimed": False,
        }
        for role in REQUIRED_ROLES:
            records.append(
                blocked_record(
                    role,
                    blocked_by="input_validation",
                    reason=(
                        "governed input validation failed"
                    ),
                )
            )

    canonical_environment: (
        dict[str, str] | None
    ) = None
    site_packages: Path | None = None
    target_libraries: list[Path] = []

    if (
        input_validation["status"] == "PASSED"
        and input_root is not None
    ):
        independent_probes = (
            (
                "base_python_runtime",
                [
                    sys.executable,
                    "-c",
                    (
                        "import platform;"
                        "print(platform.python_version())"
                    ),
                ],
                30.0,
            ),
            (
                "base_venv_import",
                [
                    sys.executable,
                    "-c",
                    "import venv;print('ok')",
                ],
                30.0,
            ),
            (
                "base_pip_import",
                [
                    sys.executable,
                    "-c",
                    (
                        "import json,pip;"
                        "print(json.dumps({"
                        "'version':pip.__version__,"
                        "'module_file':pip.__file__}))"
                    ),
                ],
                30.0,
            ),
            (
                "base_distribution_snapshot_before",
                [
                    sys.executable,
                    "-c",
                    DISTRIBUTION_SNAPSHOT_SCRIPT,
                ],
                60.0,
            ),
            (
                "gpu_topology",
                [
                    "nvidia-smi",
                    (
                        "--query-gpu=index,name,memory.total,"
                        "compute_cap,driver_version"
                    ),
                    "--format=csv,noheader,nounits",
                ],
                30.0,
            ),
        )
        for role, argv, timeout in independent_probes:
            records.append(
                run_probe(
                    role,
                    argv,
                    timeout=timeout,
                )
            )
        apply_semantic_checks(
            records,
            input_root,
        )

        append_dependent_probe(
            records,
            "venv_create_without_pip",
            (
                "base_python_runtime",
                "base_venv_import",
            ),
            [
                sys.executable,
                "-m",
                "venv",
                "--without-pip",
                str(VENV_ROOT),
            ],
            timeout=180.0,
            blocked_reason=(
                "base Python or venv prerequisite failed"
            ),
        )

        python = VENV_ROOT / "bin" / "python"
        append_dependent_probe(
            records,
            "target_runtime_identity_before_install",
            ("venv_create_without_pip",),
            controlled_target_argv(
                python,
                TARGET_IDENTITY_SCRIPT,
                str(VENV_ROOT),
            ),
            timeout=60.0,
            blocked_reason=(
                "target environment creation failed"
            ),
        )
        apply_semantic_checks(
            records,
            input_root,
        )

        append_dependent_probe(
            records,
            "offline_hash_locked_install_via_base_pip_target_directory",
            (
                "base_pip_import",
                "base_distribution_snapshot_before",
                "target_runtime_identity_before_install",
            ),
            [
                sys.executable,
                "-m",
                "pip",
                "--isolated",
                "--disable-pip-version-check",
                "install",
                "--no-index",
                "--no-cache-dir",
                "--no-deps",
                "--ignore-installed",
                "--find-links",
                str(input_root / "wheels"),
                "--require-hashes",
                "--target",
                str(target_site_packages()),
                "-r",
                str(
                    input_root
                    / "requirements.lock.txt"
                ),
            ],
            timeout=2400.0,
            blocked_reason=(
                "base pip target-directory installation failed"
            ),
        )

        install_dependency = (
            "offline_hash_locked_install_via_base_pip_target_directory",
        )
        append_dependent_probe(
            records,
            "target_distribution_inventory",
            install_dependency,
            controlled_target_argv(
                python,
                TARGET_INVENTORY_SCRIPT,
            ),
            timeout=120.0,
            blocked_reason=(
                "offline target installation did not pass"
            ),
        )
        append_dependent_probe(
            records,
            "target_dependency_check_via_controlled_python",
            install_dependency,
            controlled_target_argv(
                python,
                TARGET_DEPENDENCY_CHECK_SCRIPT,
                str(target_site_packages()),
            ),
            timeout=180.0,
            blocked_reason=(
                "offline target installation did not pass"
            ),
        )
        apply_semantic_checks(
            records,
            input_root,
        )

        if (
            dependency_failure(
                records,
                install_dependency,
            )
            is None
        ):
            site_packages = target_site_packages()
            target_libraries = (
                target_nvidia_library_directories(
                    site_packages
                )
            )
            try:
                canonical_environment, environment_record = (
                    canonical_target_environment(
                        target_libraries
                    )
                )
                records.append(environment_record)
                records.append(
                    library_inventory_record(
                        site_packages,
                        target_libraries,
                    )
                )
            except Exception as exc:
                records.append(
                    {
                        "schema_version": "3.0.0",
                        "command_role": (
                            "canonical_process_environment"
                        ),
                        "status": "FAILED",
                        "error_type": type(exc).__name__,
                        "error_message": sanitize(str(exc)),
                        "returncode": None,
                        "timed_out": False,
                        "stdout_excerpt": "",
                        "stderr_excerpt": "",
                    }
                )
                records.append(
                    blocked_record(
                        "target_cuda_library_inventory",
                        blocked_by=(
                            "canonical_process_environment"
                        ),
                        reason=(
                            "canonical target environment failed"
                        ),
                    )
                )
        else:
            records.append(
                blocked_record(
                    "canonical_process_environment",
                    blocked_by=(
                        "offline_hash_locked_install_via_base_pip_target_directory"
                    ),
                    reason=(
                        "offline target installation did not pass"
                    ),
                )
            )
            records.append(
                blocked_record(
                    "target_cuda_library_inventory",
                    blocked_by=(
                        "canonical_process_environment"
                    ),
                    reason=(
                        "canonical target environment unavailable"
                    ),
                )
            )

        apply_semantic_checks(
            records,
            input_root,
        )

        if (
            site_packages is not None
            and target_libraries
        ):
            cusparse = (
                site_packages
                / "nvidia"
                / "cusparse"
                / "lib"
                / "libcusparse.so.12"
            )
            nvjitlink = (
                site_packages
                / "nvidia"
                / "nvjitlink"
                / "lib"
                / "libnvJitLink.so.12"
            )
        else:
            cusparse = Path("/unavailable")
            nvjitlink = Path("/unavailable")

        append_dependent_probe(
            records,
            "inherited_nvjitlink_resolution",
            ("target_cuda_library_inventory",),
            ["ldd", str(cusparse)],
            timeout=120.0,
            blocked_reason=(
                "target CUDA library inventory failed"
            ),
            environment=base_offline_environment(),
        )
        append_dependent_probe(
            records,
            "canonical_nvjitlink_resolution",
            (
                "canonical_process_environment",
                "target_cuda_library_inventory",
            ),
            ["ldd", str(cusparse)],
            timeout=120.0,
            blocked_reason=(
                "canonical target environment failed"
            ),
            environment=canonical_environment,
        )
        append_dependent_probe(
            records,
            "canonical_nvjitlink_symbol",
            ("target_cuda_library_inventory",),
            [
                "readelf",
                "-Ws",
                str(nvjitlink),
            ],
            timeout=120.0,
            blocked_reason=(
                "target CUDA library inventory failed"
            ),
            environment=canonical_environment,
        )
        append_dependent_probe(
            records,
            "canonical_cusparse_direct_load",
            (
                "canonical_nvjitlink_resolution",
                "canonical_nvjitlink_symbol",
            ),
            controlled_target_argv(
                python,
                (
                    "import ctypes,sys;"
                    "ctypes.CDLL("
                    "sys.argv[1],"
                    "mode=ctypes.RTLD_GLOBAL"
                    ");print('loaded')"
                ),
                str(cusparse),
            ),
            timeout=120.0,
            blocked_reason=(
                "canonical loader proof did not pass"
            ),
            environment=canonical_environment,
        )
        append_dependent_probe(
            records,
            "target_process_environment",
            ("canonical_process_environment",),
            controlled_target_argv(
                python,
                TARGET_PROCESS_SCRIPT,
            ),
            timeout=60.0,
            blocked_reason=(
                "canonical target environment failed"
            ),
            environment=canonical_environment,
        )
        apply_semantic_checks(
            records,
            input_root,
        )

        runtime_dependencies = (
            "target_distribution_inventory",
            "target_dependency_check_via_controlled_python",
            "canonical_cusparse_direct_load",
            "target_process_environment",
            "gpu_topology",
        )
        append_dependent_probe(
            records,
            "python_runtime",
            runtime_dependencies,
            controlled_target_argv(
                python,
                (
                    "import platform;"
                    "print(platform.python_version())"
                ),
            ),
            timeout=30.0,
            blocked_reason=(
                "canonical runtime prerequisites failed"
            ),
            environment=canonical_environment,
        )
        append_dependent_probe(
            records,
            "torch_family_runtime",
            runtime_dependencies,
            controlled_target_argv(
                python,
                (
                    "import json,torch,torchaudio,torchvision;"
                    "print(json.dumps({"
                    "'torch':torch.__version__,"
                    "'torchaudio':torchaudio.__version__,"
                    "'torchvision':torchvision.__version__,"
                    "'cuda':torch.version.cuda,"
                    "'available':torch.cuda.is_available(),"
                    "'device_count':torch.cuda.device_count()"
                    "}))"
                ),
            ),
            timeout=180.0,
            blocked_reason=(
                "canonical runtime prerequisites failed"
            ),
            environment=canonical_environment,
        )
        append_dependent_probe(
            records,
            "transformers_runtime",
            (
                "target_distribution_inventory",
                "target_process_environment",
            ),
            controlled_target_argv(
                python,
                (
                    "import transformers;"
                    "print(transformers.__version__)"
                ),
            ),
            timeout=60.0,
            blocked_reason=(
                "target environment prerequisites failed"
            ),
            environment=canonical_environment,
        )
        append_dependent_probe(
            records,
            "vllm_distribution",
            ("target_distribution_inventory",),
            controlled_target_argv(
                python,
                (
                    "import importlib.metadata;"
                    "print("
                    "importlib.metadata.version('vllm')"
                    ")"
                ),
            ),
            timeout=60.0,
            blocked_reason=(
                "target inventory did not pass"
            ),
            environment=canonical_environment,
        )
        append_dependent_probe(
            records,
            "vllm_module",
            (
                "torch_family_runtime",
                "transformers_runtime",
                "vllm_distribution",
            ),
            controlled_target_argv(
                python,
                (
                    "import vllm;"
                    "print(vllm.__version__)"
                ),
            ),
            timeout=180.0,
            blocked_reason=(
                "Torch or vLLM prerequisite failed"
            ),
            environment=canonical_environment,
        )
        append_dependent_probe(
            records,
            "vllm_native_extension",
            ("vllm_module",),
            controlled_target_argv(
                python,
                (
                    "import importlib;"
                    "importlib.import_module('vllm._C');"
                    "print('ok')"
                ),
            ),
            timeout=180.0,
            blocked_reason=(
                "vLLM module import did not pass"
            ),
            environment=canonical_environment,
        )

        records.append(
            run_probe(
                "base_distribution_snapshot_after",
                [
                    sys.executable,
                    "-c",
                    DISTRIBUTION_SNAPSHOT_SCRIPT,
                ],
                timeout=60.0,
            )
        )
        apply_semantic_checks(
            records,
            input_root,
        )

    states = summarize_role_states(records)
    observed = {
        str(record["command_role"]): record
        for record in records
    }
    first_divergence = next(
        (
            role
            for role in REQUIRED_ROLES
            if observed[role]["status"] == "FAILED"
        ),
        None,
    )
    install_record = observed[
        "offline_hash_locked_install_via_base_pip_target_directory"
    ]
    package_installation_started = (
        install_record["status"]
        in {"PASSED", "FAILED"}
    )

    before_snapshot = observed.get(
        "base_distribution_snapshot_before"
    )
    after_snapshot = observed.get(
        "base_distribution_snapshot_after"
    )
    base_distribution_metadata_unchanged = (
        before_snapshot is not None
        and after_snapshot is not None
        and before_snapshot["status"] == "PASSED"
        and after_snapshot["status"] == "PASSED"
        and parse_json_stdout(before_snapshot)
        == parse_json_stdout(after_snapshot)
    )
    target_identity = observed.get(
        "target_runtime_identity_before_install"
    )
    target_environment_isolated = (
        target_identity is not None
        and target_identity["status"] == "PASSED"
    )
    target_process = observed.get(
        "target_process_environment"
    )
    target_process_environment_isolated = (
        target_process is not None
        and target_process["status"] == "PASSED"
    )
    canonical_resolution = observed.get(
        "canonical_nvjitlink_resolution"
    )
    canonical_nvjitlink_resolved_to_target = (
        canonical_resolution is not None
        and canonical_resolution["status"] == "PASSED"
    )
    symbol = observed.get(
        "canonical_nvjitlink_symbol"
    )
    required_nvjitlink_symbol_present = (
        symbol is not None
        and symbol["status"] == "PASSED"
    )

    write_evidence_record(
        "00_input_validation.json",
        input_validation,
    )
    for index, role in enumerate(
        REQUIRED_ROLES,
        start=1,
    ):
        write_evidence_record(
            f"10_{index:02d}_{role}.json",
            observed[role],
        )

    passed = (
        input_validation["status"] == "PASSED"
        and not states["failed_required_roles"]
        and not states["blocked_required_roles"]
        and not states["not_executed_required_roles"]
        and base_distribution_metadata_unchanged
        and target_environment_isolated
        and target_process_environment_isolated
        and canonical_nvjitlink_resolved_to_target
        and required_nvjitlink_symbol_present
    )
    summary = {
        "schema_version": "7.0.0",
        "diagnostic_id": NOTEBOOK_NAME,
        "captured_at": datetime.now(UTC).isoformat(
            timespec="seconds"
        ),
        "status": "PASSED" if passed else "FAILED",
        "first_divergence": first_divergence,
        **states,
        "installation_executor": (
            "BASE_PIP_TARGET_DIRECTORY"
        ),
        "dependency_validation": (
            DEPENDENCY_VALIDATION_POLICY
        ),
        "canonical_loader_policy": (
            "TARGET_NVIDIA_LIBRARIES_PREPENDED"
        ),
        "python_startup_policy": (
            "NO_SITE_WITH_CONTROLLED_SITE_BOOTSTRAP"
        ),
        "input_sha256_manifest_sha256": (
            EXPECTED_HASHES[
                "sha256_manifest.json"
            ]
        ),
        "input_materialization_receipt_sha256": (
            EXPECTED_HASHES[
                "materialization_receipt.json"
            ]
        ),
        "input_runtime_manifest_sha256": (
            EXPECTED_HASHES[
                "runtime_manifest.json"
            ]
        ),
        "resolution_lock_sha256": (
            EXPECTED_HASHES[
                "resolution_lock.json"
            ]
        ),
        "package_count": EXPECTED_PACKAGE_COUNT,
        "package_installation_started": (
            package_installation_started
        ),
        "target_environment_isolated": (
            target_environment_isolated
        ),
        "target_process_environment_isolated": (
            target_process_environment_isolated
        ),
        "canonical_nvjitlink_resolved_to_target": (
            canonical_nvjitlink_resolved_to_target
        ),
        "required_nvjitlink_symbol_present": (
            required_nvjitlink_symbol_present
        ),
        "base_distribution_metadata_unchanged": (
            base_distribution_metadata_unchanged
        ),
        "network_access_requested": False,
        "model_requests_performed": 0,
        "benchmark_trajectory_requests_performed": 0,
        "qualification_claimed": False,
        "credentials_used": False,
        "customer_data_used": False,
        "external_spend": 0,
    }
    write_evidence_record(
        "90_summary.json",
        summary,
    )

    payload_paths = tuple(
        sorted(
            EVIDENCE_ROOT.iterdir(),
            key=lambda path: path.name,
        )
    )
    sha_entries = [
        {
            "path": path.name,
            "sha256": streaming_sha256(path),
            "size_bytes": path.stat().st_size,
        }
        for path in payload_paths
    ]
    write_evidence_record(
        "99_evidence_sha256.json",
        {
            "schema_version": "1.0.0",
            "entries": sha_entries,
        },
    )

    with zipfile.ZipFile(
        OUTPUT_ZIP,
        "w",
        compression=zipfile.ZIP_DEFLATED,
    ) as archive:
        for path in sorted(
            EVIDENCE_ROOT.iterdir(),
            key=lambda item: item.name,
        ):
            archive.write(
                path,
                arcname=path.name,
            )

    print(f"artifact={OUTPUT_ZIP}")
    print(
        f"size_bytes={OUTPUT_ZIP.stat().st_size}"
    )
    print(
        f"sha256={streaming_sha256(OUTPUT_ZIP)}"
    )
    print(
        "offline_compatibility_status="
        + str(summary["status"])
    )
    print(
        "dependency_validation="
        + DEPENDENCY_VALIDATION_POLICY
    )
    print(
        f"first_divergence={first_divergence}"
    )
    print(
        "failed_required_roles="
        + canonical_json(
            states["failed_required_roles"]
        )
    )
    print(
        "blocked_required_roles="
        + canonical_json(
            states["blocked_required_roles"]
        )
    )
    print(
        "not_executed_required_roles="
        + canonical_json(
            states["not_executed_required_roles"]
        )
    )
    print(
        "canonical_nvjitlink_resolved_to_target="
        + str(
            canonical_nvjitlink_resolved_to_target
        ).lower()
    )
    print(
        "target_process_environment_isolated="
        + str(
            target_process_environment_isolated
        ).lower()
    )
    print(
        "base_distribution_metadata_unchanged="
        + str(
            base_distribution_metadata_unchanged
        ).lower()
    )
    print("model_requests_performed=0")
    print("qualification_claimed=false")
    print("upload_only_this_file=true")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
